# FFS scan of raw OCM frames

Runs the full `pyaraucaria.ffs` pipeline (the steps of `calc_frame_fwhm`, with `LINES_NSIGMA` for lines) (branch `mgorski_dev`) on the raw frames
(`SOURCE = "raw"`) or the reduced ofp frames (`SOURCE = "zdf"`, science only) of one telescope and
stores, per frame: file name, selected header cards (+ the full header as text), all scalar FFS frame
statistics, line / star counts and the Moon/Sun geometry. No maps or masks are saved.

The database is one directory of parquet files per telescope and source (`DB_DIR/<telescope>/*.parquet` for raw,
`DB_DIR/<telescope>_zdf/*.parquet` for zdf), written
in chunks, so an interrupted run loses at most one chunk. On the next run frames already in the
database are skipped: the scan goes from the newest frame backwards, so it first picks up new frames and
then continues into older nights.

Analysis of the database: separate notebook, `load_db()` below reads it into a `DataFrame`.

In [1]:
# configuration

import os

TELESCOPE = "zb08"            # wk06, zb08, jk15
SOURCE = "zdf"                # "raw" - raw/<night>/*.fits, "zdf" - processed-ofp/science/<night>/<frame>/*_zdf.fits
N_NIGHTS = 12                  # how many nights back from today (1 = last night only)
FITS_DIR = "/work/vela/oca/fits"
DB_DIR = os.path.expanduser("~/data/ffs_db")

IMAGETYPES = None             # None = all frame types, or e.g. ("science",)
MAX_FILES = None              # stop after this many measured frames (None = no limit)
CHUNK = 20                    # frames per parquet file
RETRY_ERRORS = False          # measure again frames that failed last time

# FFS parameters
SATURATION = {"zb08": 60000, "wk06": 60000, "jk15": 150000}  # ADU, raw; SATURATE is empty in the headers, jk15 is BITPIX 32 with star peaks up to ~170k; used also for zdf
AUTOBIN = {"zb08": False, "wk06": False, "jk15": True}       # jk15 4k: ~27 s/frame at bin 1, ~15 s with autobin
LINES_NSIGMA = 5              # threshold mask for line detection; FFS default 3 gives false lines on galaxies and dense fields
N_STARS = 100                 # brightest stars measured
STARS_MASK_K = 3              # second pass: stats and sky without discs of k * FWHM around stars

# header cards stored as columns (the full header is stored as text anyway)
HDR_KEYS = ["DATE-OBS", "JD", "IMAGETYP", "OBSTYPE", "OBJECT", "FILTER", "EXPTIME", "TEST",
            "RA_TEL", "DEC_TEL", "ALT_TEL", "AZ_TEL", "AIRMASS", "FOCUS", "ROTATOR",
            "T-CAM", "T-CAMSET", "READ-MOD", "GAIN-MOD", "GAIN", "RON", "SCALE",
            "DOME-SHT", "DOME-AZ", "MIRR-COV", "TRACKING", "FAN-MIRR", "FAN-DOME",
            "T-WS", "P-WS", "RHUM-WS", "WIND-DIR", "WIND-AVG", "WIND-GUS",
            "M1-POS1", "M1-POS2", "M1-POS3", "NAXIS1", "NAXIS2",
            "OFP_DATE", "OFP_ZERO", "OFP_DARK", "OFP_FLAT"]   # OFP_* only in zdf

In [2]:
import time
import glob
import warnings
import subprocess

import numpy as np
import pandas as pd
import astropy.units as u
from astropy.io import fits
from astropy.time import Time
from astropy.coordinates import SkyCoord, EarthLocation, AltAz, get_sun, get_body

import pyaraucaria
from pyaraucaria.ffs import FFS

# which ffs measured a frame: FFS on mgorski_dev changes, results of different versions may not be comparable
_repo = os.path.dirname(os.path.dirname(pyaraucaria.__file__))
_commit = subprocess.run(["git", "-C", _repo, "describe", "--always", "--dirty"], capture_output=True, text=True).stdout.strip()
FFS_VERSION = _commit
print("pyaraucaria", FFS_VERSION, pyaraucaria.__file__)

pyaraucaria v2.13.0-57-g6d911ce /home/mgorski/src/pyaraucaria/pyaraucaria/__init__.py


In [3]:
def current_night():
    """OCM night number (directory name in raw/) of the coming / ongoing night: JD - 2460000 at noon UTC."""
    return int(Time.now().jd) - 2460000


def list_files(telescope, n_nights, source):
    """Raw ("raw") or reduced ("zdf") frames of the last n_nights nights, newest first."""
    root = os.path.join(FITS_DIR, telescope, "raw" if source == "raw" else "processed-ofp/science")
    pattern = "*.fits" if source == "raw" else "*/*_zdf.fits"
    first = current_night() - n_nights + 1
    nights = sorted((d for d in os.listdir(root) if d.isdigit() and int(d) >= first), reverse=True)  # date-named dirs in raw/ are symlinks to these
    files = []
    for n in nights:
        files += sorted(glob.glob(os.path.join(root, n, pattern)), reverse=True)
    return files


def db_name(telescope, source):
    """Directory of the database in DB_DIR: zb08 for raw, zb08_zdf for zdf."""
    return telescope if source == "raw" else f"{telescope}_{source}"


def load_db(telescope, source="raw"):
    """All rows measured for the telescope and source; for a frame measured more than once only the last row."""
    files = sorted(glob.glob(os.path.join(DB_DIR, db_name(telescope, source), "*.parquet")))
    if not files:
        return pd.DataFrame()
    db = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)
    return db.sort_values("measured_at").drop_duplicates("file", keep="last").sort_values("file", ignore_index=True)


def save_chunk(rows, telescope, source):
    """Write rows as a new parquet file in DB_DIR/db_name(telescope, source)."""
    if not rows:
        return
    path = os.path.join(DB_DIR, db_name(telescope, source))
    os.makedirs(path, exist_ok=True)
    name = os.path.join(path, Time.now().strftime("%Y%m%d_%H%M%S") + f"_{len(rows)}.parquet")
    pd.DataFrame(rows).to_parquet(name, index=False)

In [4]:
def hdr_value(hdr, key):
    """Header value; missing or empty ('') -> None."""
    v = hdr.get(key)
    return None if v is None or v == "" else v


def sky_geometry(hdr):
    """Moon and Sun altitude, Moon illumination and Moon distance from the field [deg]."""
    location = EarthLocation(lat=hdr["OBS-LAT"] * u.deg, lon=hdr["OBS-LONG"] * u.deg, height=hdr["OBS-ELEV"] * u.m)
    t = Time(hdr["DATE-OBS"], format="isot", scale="utc")
    altaz = AltAz(obstime=t, location=location)
    field = SkyCoord(ra=hdr["RA_TEL"] * u.deg, dec=hdr["DEC_TEL"] * u.deg, frame="icrs")
    sun = get_sun(t)
    moon = get_body("moon", t, location)
    elongation = moon.separation(sun, origin_mismatch="ignore").rad
    return {
        "sun_alt": sun.transform_to(altaz).alt.deg,
        "moon_alt": moon.transform_to(altaz).alt.deg,
        "moon_illum": (1 - np.cos(elongation)) / 2,
        "moon_sep": field.separation(moon, origin_mismatch="ignore").deg,
    }


def measure(path, telescope):
    """One row of the database: file, header, FFS frame statistics, lines, stars, Moon/Sun."""
    row = {"file": os.path.basename(path), "path": path, "telescope": telescope,
           "night": int(os.path.basename(path).split("_")[1]),
           "measured_at": Time.now().isot, "ffs_version": FFS_VERSION, "error": None}
    t0 = time.time()
    try:
        with fits.open(path) as hl:
            hdr = hl[0].header
            data = hl[0].data.astype(float)
        row["header"] = hdr.tostring(sep="\n")
        row.update({key: hdr_value(hdr, key) for key in HDR_KEYS})
        if IMAGETYPES is not None and hdr.get("IMAGETYP") not in IMAGETYPES:
            return None

        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)   # nanmedian of empty slices on frames without stars
            ffs = FFS(data, gain=hdr_value(hdr, "GAIN") or 1., rn_noise=hdr_value(hdr, "RON") or 0., saturation=SATURATION[telescope], autobin=AUTOBIN[telescope])
            # the steps of calc_frame_fwhm, with our own nsigma of the threshold mask
            ffs.mk_saturation_mask()
            ffs.mk_stats()
            ffs.sky_map()
            ffs.mk_cosmics_mask()
            ffs.mk_threshold_mask(nsigma=LINES_NSIGMA)
            ffs.find_lines()
            ffs.classify_lines()
            ffs.find_stars(threshold=10, fwhm=10)
            ffs.star_info(box=10, N_stars=N_STARS)
            ffs.calc_star_stats()
            if np.isfinite(ffs.frame_fwhm):
                # second pass: frame statistics and sky without the stars
                ffs.mk_stars_mask(k=STARS_MASK_K)
                ffs.mk_stats()
                ffs.sky_gradient()
            if ffs.frame_used_stars >= 5:
                ffs.calc_star_field_stats()

        for key, v in ffs.stats["frame"].items():
            row[key] = v.tolist() if isinstance(v, np.ndarray) else v
        lines = ffs.lines
        trails = lines[~lines["spike"]] if len(lines) else lines
        row["bin"] = ffs.bin
        row["n_stars"] = len(ffs.coo)
        row["n_lines"] = len(trails)
        row["n_spikes"] = len(lines) - len(trails)
        row["line_max_len"] = float(np.max(np.hypot(trails["x1"] - trails["x0"], trails["y1"] - trails["y0"]))) if len(trails) else 0.
        row["line_max_val"] = float(np.max(trails["val"])) if len(trails) else 0.
        row["fwhm_arcsec"] = row["fwhm"] * row["SCALE"] if row.get("SCALE") else None
        if hdr_value(hdr, "RA_TEL") is not None:
            row.update(sky_geometry(hdr))
    except Exception as e:
        row["error"] = f"{type(e).__name__}: {e}"
    row["ffs_time"] = time.time() - t0
    return row

In [5]:
COLS = f"{'file':26s} {'type':8s} {'object':16s} {'flt':4s} {'exp':>6s} {'sky':>7s} {'grad':>7s} {'lines':>5s} {'stars':>5s} {'fwhm':>5s} {'fw"':>5s} {'ell':>5s} {'e_t':>6s} {'moon':>5s} {'t[s]':>5s}"

def fmt(x, f):
    """Number formatted with f, or '-' when missing / NaN."""
    return format("-", ">" + f.split(".")[0]) if x is None or (isinstance(x, float) and not np.isfinite(x)) else format(x, f)


def show(r):
    """One line with the most interesting values of a database row."""
    if r["error"]:
        return f"{r['file']:26s} ERROR {r['error']}"
    return (f"{r['file']:26s} {str(r['IMAGETYP'])[:8]:8s} {str(r['OBJECT'])[:16]:16s} {str(r['FILTER'])[:4]:4s} {fmt(r['EXPTIME'], '6.1f')} "
            f"{fmt(r.get('median'), '7.0f')} {fmt(r.get('bkg_frame_gradient'), '7.1f')} {r['n_lines']:5d} {r['n_stars']:5d} "
            f"{fmt(r.get('fwhm'), '5.2f')} {fmt(r.get('fwhm_arcsec'), '5.2f')} {fmt(r.get('ellipticity'), '5.2f')} {fmt(r.get('psf_e_t'), '6.3f')} "
            f"{fmt(r.get('moon_alt'), '5.0f')} {r['ffs_time']:5.1f}")

## Scan

Columns: `sky` – background median [ADU] (without stars), `grad` – `bkg_frame_gradient` [ADU per frame],
`lines` – satellite / plane trails (diffraction spikes not counted), `stars` – all detections,
`fwhm` [px] and `fw"` [arcsec], `ell` – median ellipticity, `e_t` – tangential pattern of ellipticity
(> 0 tangential, < 0 radial), `moon` – Moon altitude [deg], `t[s]` – time per frame.

The loop can be stopped at any moment (interrupt the kernel): what is measured so far is saved.

In [6]:
db = load_db(TELESCOPE, SOURCE)
done = set(db["file"]) if len(db) else set()
if RETRY_ERRORS and len(db):
    done -= set(db.loc[db["error"].notna(), "file"])

files = list_files(TELESCOPE, N_NIGHTS, SOURCE)
todo = [f for f in files if os.path.basename(f) not in done]
print(f"{TELESCOPE} {SOURCE}: {len(files)} frames in the last {N_NIGHTS} nights, {len(files) - len(todo)} already in the database, {len(todo)} to measure")

rows = []
n = 0
print(COLS)
try:
    for f in todo:
        if MAX_FILES is not None and n >= MAX_FILES:
            break
        r = measure(f, TELESCOPE)
        if r is None:
            continue
        print(show(r))
        rows.append(r)
        n += 1
        if len(rows) >= CHUNK:
            save_chunk(rows, TELESCOPE, SOURCE)
            rows = []
finally:
    save_chunk(rows, TELESCOPE, SOURCE)
    print("DONE")

zb08 zdf: 1096 frames in the last 12 nights, 0 already in the database, 1096 to measure
file                       type     object           flt     exp     sky    grad lines stars  fwhm   fw"   ell    e_t  moon  t[s]
zb08c_1325_77965_zdf.fits  science  SN2025pht        V     250.0       3     0.3     0    35     -     -     -      -   -44   2.4
zb08c_1325_77666_zdf.fits  science  SN2025pht        V     250.0     302   105.5     1   275  3.88  1.96  0.03 -0.006   -45   2.4
zb08c_1325_77358_zdf.fits  science  SN2025pht        V     250.0     416   168.6     1   307  3.82  1.92  0.06 -0.008   -45   2.3
zb08c_1325_77053_zdf.fits  science  SN2025pht        B     250.0     172    77.1     1   259  4.04  2.04  0.05 -0.012   -45   2.6
zb08c_1325_76750_zdf.fits  science  SN2025pht        B     250.0     171    91.4     3   283  3.99  2.01  0.05 -0.006   -46   2.9
zb08c_1325_76447_zdf.fits  science  SN2025pht        B     250.0     172    82.0     1   263  3.89  1.96  0.05 -0.006   -46   2.5
zb

KeyboardInterrupt: 

## Database

Everything measured so far for the telescope and source.

In [ ]:
db = load_db(TELESCOPE, SOURCE)
assert len(db), f"no {SOURCE} frames of {TELESCOPE} in {DB_DIR}"
print(len(db), "frames,", db["error"].notna().sum(), "errors,", db["night"].nunique(), "nights:", sorted(db["night"].unique().tolist()))
db[["file", "IMAGETYP", "OBJECT", "FILTER", "EXPTIME", "median", "bkg_frame_gradient", "n_lines", "n_stars", "fwhm", "ellipticity", "moon_alt"]].tail(20)